In [ ]:
import flow
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import shutil
import matplotlib.pyplot as plt
from flow import FlowProject, directives
from pymser import pymser
import warnings
import panedr

In [ ]:
file = "example_output_files/sft_calc.edr"
edr = panedr.edr_to_df(file)

In [ ]:
edr.tail()

In [ ]:
dt = edr["Time"].values
print(dt)
nsteps = 70000000
tot_rows = 70000
prod_step = 10000
nprod = tot_rows*prod_step
print(nprod)
print(nsteps/tot_rows)
print(nsteps/prod_step)

In [ ]:
len(edr)

In [ ]:
np.mean(edr["Pressure"].values)

In [ ]:
# Get the density and volume data
sim_name = "eql2"
df_all = edr
eq_data_dict = {}
property = "Density"
# if property not in df_all.columns:
#     print("A")
#     pass
    # df = df[["Time", property]].copy()
    # print(df.head())

if property in ["Total-Energy", "Volume", "Pressure", "Density", "#Surf*SurfTen"]:
    # command = "source /afs/crc.nd.edu/group/maginn/group_members/Ryan_DeFever/software/gromacs-2020/gromacs-dp/bin/GMXRC"
    # subprocess.run(command, shell=True, check=True)
    # print(os.path.exists(os.path.join("example_output_files", f"{sim_name}.edr")))
    # command = (
    #     f"source /afs/crc.nd.edu/group/maginn/group_members/Ryan_DeFever/software/gromacs-2020/gromacs-dp/bin/GMXRC && module load gromacs && gmx_d energy -f example_output_files/{sim_name}.edr -s example_output_files/{sim_name}.tpr -o {sim_name}_{property}.xvg"
    # )
    # subprocess.run(command, input=f"{property}\n", text=True, check=True, shell = True)
    prop_data = np.loadtxt(
        sim_name + "_" + property + ".xvg", comments=["#", "@"]
    )

    df = pd.DataFrame(prop_data)

property_data = df.iloc[:, 1].values
plt.plot(df.iloc[:, 0].values, df.iloc[:, 1].values)
time_data = df.iloc[:, 0].values
eq_col_file = sim_name + "_" + property + ".csv"
eq_data_dict[property] = {
    "data": property_data,
    "time_data": time_data,
    "file": eq_col_file,
}

In [ ]:
gro_file = "example_output_files/sft_calc.gro"
# Get final box volume
with open(gro_file, "rb") as f:
    # Move the pointer to the end of the file, but leave space to find the last line
    f.seek(-2, os.SEEK_END)
    # Read backward until a newline is found
    while f.read(1) != b"\n":
        f.seek(-2, os.SEEK_CUR)
    # Read the last line after finding the newline
    last_line = f.readline().decode()
last_line.strip()
ave_length = list(map(float, last_line.split()))[0]

print(ave_length)

In [ ]:
def plot_res_pymser(t_col, eq_col, results, name):
    fig, [ax1, ax2] = plt.subplots(
        1, 2, gridspec_kw={"width_ratios": [2, 1]}, sharey=True
    )

    ax1.set_ylabel(name, color="black", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Time (ns)", fontsize=14, fontweight="bold")

    ax1.plot(t_col, eq_col, label="Raw data", color="blue")

    ax1.plot(
        t_col[results["t0"] :],
        results["equilibrated"],
        label="Equilibrated data",
        color="red",
    )

    ax1.plot(
        [0, t_col[-1]],
        [results["average"], results["average"]],
        color="green",
        zorder=4,
        label="Equilibrated average",
    )

    ax1.fill_between(
        t_col,
        results["average"] - results["uncertainty"],
        results["average"] + results["uncertainty"],
        color="lightgreen",
        alpha=0.3,
        zorder=4,
    )

    # ax1.set_yticks(np.arange(eq_col.min(), eq_col.max(), eq_col.max()/10))
    ax1.set_xlim(t_col.min(), t_col.max())
    ax1.tick_params(axis="y", labelcolor="black")

    ax1.grid(alpha=0.3)
    ax1.legend()

    ax2.hist(
        eq_col,
        orientation="horizontal",
        bins=30,
        edgecolor="blue",
        lw=1.5,
        facecolor="white",
        zorder=3,
    )

    bin_red = 10
    ax2.hist(
        results["equilibrated"],
        orientation="horizontal",
        bins=bin_red,
        edgecolor="red",
        lw=1.5,
        facecolor="white",
        zorder=3,
    )

    ymax = int(ax2.get_xlim()[-1])

    ax2.plot(
        [0, ymax],
        [results["average"], results["average"]],
        color="green",
        zorder=4,
        label="Equilibrated average",
    )

    ax2.fill_between(
        range(ymax),
        results["average"] - results["uncertainty"],
        results["average"] + results["uncertainty"],
        color="lightgreen",
        alpha=0.3,
        zorder=4,
    )

    ax2.set_xlim(0, ymax)

    ax2.grid(alpha=0.5, zorder=1)

    fig.set_size_inches(9, 5)
    fig.set_dpi(100)
    fig.tight_layout()
    # save_name = "MSER_eq_vol.png"
    # fig.savefig(job.fn(save_name), dpi=300, facecolor="white")
    # plt.close(fig)

def check_equil_converge(eq_data_dict, prod_tol):
    equil_matrix = []
    res_matrix = []
    prop_names = list(eq_data_dict.keys())
    num_cols = len(prop_names)

    try:
        # Load data for both boxes
        for key in list(eq_data_dict.keys()):
            eq_col = eq_data_dict[key]["data"]
            prod_tol = len(eq_col)/8
            print(prod_tol)
            batch_size = max(1, int(len(eq_col) * 0.0005))

            # Try with ADF test enabled, fallback without it if it fails
            try:
                results = pymser.equilibrate(
                    eq_col,
                    LLM=False,
                    batch_size=batch_size,
                    ADF_test=True,
                    uncertainty="uSD",
                    print_results=False,
                )
                adf_test_failed = results["critical_values"]["1%"] <= results["adf"]
            except:
                results = pymser.equilibrate(
                    eq_col,
                    LLM=False,
                    batch_size=batch_size,
                    ADF_test=False,
                    uncertainty="uSD",
                    print_results=False,
                )
                results["adf"], results["critical_values"], adf_test_failed = (
                    None,
                    None,
                    False,
                )

            equilibrium = len(eq_col) - results["t0"] >= prod_tol
            equil_matrix.append(equilibrium and not adf_test_failed)
            res_matrix.append(results)

        for i, is_equilibrated in enumerate(equil_matrix):
            key_name = list(eq_data_dict.keys())[i]
            col_vals = eq_data_dict[key_name]["data"]
            t_vals = eq_data_dict[key_name]["time_data"]
            print(t_vals.min(), t_vals.max())
            # plot all

            # if not all(equil_matrix):
            plot_res_pymser(
                t_vals, col_vals, res_matrix[i], prop_names[i % num_cols]
            )

            # Display outcome
            prod_cycles = len(col_vals) - res_matrix[i]["t0"]
            if is_equilibrated:
                # Plot successful equilibration
                statement = f"       > Success! Found {prod_cycles} production cycles."
            else:
                # Plot failed equilibration
                statement = f"       > Equil Failure! "
                if res_matrix[i]["adf"] is None:
                    # Note: ADF test failed to complete
                    statement += f"ADF test failed to complete! "
                elif res_matrix[i]["adf"] > res_matrix[i]["critical_values"]["1%"]:
                    adf, one_pct = (
                        res_matrix[i]["adf"],
                        res_matrix[i]["critical_values"]["1%"],
                    )
                    statement += f"ADF value: {adf}, 99% confidence value: {one_pct}! "
                if len(col_vals) - res_matrix[i]["t0"] < prod_tol:
                    statement += f"Only {prod_cycles} production cycles found."
            print(statement)
            return statement
            # with open("Equil_Output.txt", "a") as f:
            #     print(statement, file=f)

    except Exception as e:
        # This will cause an error in the GEMC operation which lets us know that the job failed
        raise Exception("Error processing job ")



In [ ]:

vals = check_equil_converge(eq_data_dict, prod_tol = None)


In [ ]:
from thermo import Chemical
from thermo import *
import unyt as u
from utils.molec_class_files import esolvs

# Load class properies for each training molecule
name = "DMF"
mol_names = [name]  # , "Gly", "ACN", "MeOH", "DMSO", "THF", "DCM", "DEC"]
molec_dict = esolvs.make_dict(mol_names)
molec_data = molec_dict[name]
# from thermo.eos import PengRobinson

# Define the compound (e.g., 'methane', 'ethane', etc.)
smiles_string =  molec_data.smiles_str # Replace with your compound of interest
T = list(molec_data.expt_Pvap.keys())[0]  # Temperature in Kelvin (adjust as needed)
P = list(molec_data.expt_Pvap.values())[0] # Pressure in Pascals (adjust as needed)
P = float((P*u.bar).in_units(u.Pa).value)
# Create a Chemical object for the compound
constants = Chemical(smiles_string)
# print(constants)
# eos_kwargs = {'Pc': constants.Pc, 'Tc': constants.Tc, 'omega': constants.omega}

eos = PR(constants.Tc, constants.Pc, constants.omega, T=T, P=P)

# Use the Peng-Robinson EOS to calculate the vapor density
molar_vol_g = eos.V_g
vapor_density = molec_data.molecular_weight/1000/ molar_vol_g
# Print the vapor density
print(f'PR Vapor Density of {constants.name} at {T} K and {P} Pa: rho_v = {vapor_density} kg/m^3')

unit_T = T * u.K
mw = molec_data.molecular_weight * (u.g / u.mol)
R = 8.314 * (u.Pa * u.m**3) / (u.mol * u.K)
vapor_density = float(
    ((molec_data.expt_Pvap[T] * u.bar * mw) / (R * unit_T)).in_units("kg/m**3").value
)
print(f'IG Vapor Density of {constants.name} at {T} K and {P} Pa: rho_v = {vapor_density} kg/m^3')